# Changelog classifier

Checks how the monthly changes are classified **before** anything goes to HubSpot. All rules live in `core.master_continuity` and
`core.master_lifecycle` (the master build runs them); this notebook only calls them and shows the result. Nothing is written.

The changelog knows three kinds: **NEW**, **CHANGED**, **RETIRED** (plus REACTIVATED for a retired site that is listed again).
- A site that got a new source id at its address (**rekeyed**) or moved within its town (**moved**: near, and the same name,
  phone or a rare e-mail / website domain) keeps its PS-ID: CHANGED.
- A new listing at an address its source already has joins that row: no NEW.
- A site is RETIRED only after `RETIRE_AFTER_MISSED_RUNS` (2) fresh runs in a row without it; a stale or rejected run neither
  counts nor resets (`missed_runs` column). A duplicate of another row is closed as CHANGED ("duplicate of ...").

Order: 1 pick two masters, 2 candidate pairs (all features), 3 the pairs the build would take, 4 what is left, 5 one master's
changelog as the build classified it. Personal data is in the outputs: keep them out of git and out of chats.

## 1. Two masters

In [ ]:
from pathlib import Path
import pandas as pd
from core import master_continuity as mc, master_lifecycle as ml, master_schema as ms, paths, hubspot_export as hx
from core.master_run import read_master
pd.set_option('display.width', 220, 'display.max_colwidth', 60, 'display.max_rows', 300)

masters = paths.official_masters()
print('official masters:', [p.parent.name for p in masters])
if len(masters) < 2:  # the newest master is built in Actions and lives on the Drive: a downloaded copy of the next month
    y, m = map(int, masters[-1].parent.name.split('-'))
    masters.append(Path.home() / 'Downloads' / f'pruefstellen_master_{y + m // 12}-{m % 12 + 1:02d}.csv')
PREV, CUR = masters[-2], masters[-1]  # or two paths of your own
prev, cur_all = read_master(PREV), read_master(CUR)
month = CUR.stem.rsplit('_', 1)[1]
cur = cur_all[cur_all['last_seen'].str.startswith(month)]  # the rows this run found
print(PREV.name, len(prev), '->', CUR.name, len(cur_all), f'({len(cur)} found this run)')
print('RETIRE_AFTER_MISSED_RUNS =', ms.RETIRE_AFTER_MISSED_RUNS, '| REKEY_KM =', mc.REKEY_KM, '| MOVE_KM =', mc.MOVE_KM, '| NAME_MIN =', mc.NAME_MIN)

## 2. Candidate pairs
Every (missing old row, new row) pair of one ÜO within `MOVE_KM` or the same town, with the features the rules look at.
`verdict` empty = no evidence: the old row stays missing, the new row stays NEW.
On a master built with these rules the pairs are already joined, so compare a master built WITHOUT them (e.g. October -> the
November master of 2026-11-02) to see what the rules would change.

In [ ]:
cand = mc.candidates(prev, cur)
print(len(cand), 'candidate pairs |', cand['verdict'].value_counts().to_dict())
cand.sort_values(['verdict', 'km'], ascending=[False, True])

## 3. The pairs the build takes (1:1, best first)

In [ ]:
taken = pd.DataFrame(mc.match(prev, cur))
taken.merge(cand[['old', 'new', 'old_name', 'new_name', 'old_address', 'new_address']], on=['old', 'new'], how='left') if len(taken) else taken

## 4. What is left
Missing rows without a pair (they count a fresh miss; retired at the second) and new rows without a pair (NEW).
Look these up by hand when the list looks odd (a finder gap shows up here as a whole region missing).

In [ ]:
used_old, used_new = set(taken.get('old', [])), set(taken.get('new', []))
p = ml.with_lifecycle_columns(prev)
missing = p[(p['status'] == 'active') & ~p['master_id'].isin(set(cur['master_id'])) & ~p['master_id'].isin(used_old)]
new = cur[~cur['master_id'].isin(set(prev['master_id'])) & ~cur['master_id'].isin(used_new)]
cols = ['master_id', 'uo', 'name', 'street', 'zip', 'city', 'last_seen', 'missed_runs', 'source_ids']
print(len(missing), 'missing without a pair |', len(new), 'new without a pair')
display(missing[[c for c in cols if c in missing.columns]].sort_values(['uo', 'city']))
display(new[[c for c in cols if c in new.columns]].sort_values(['uo', 'city']))

## 5. The changelog as the build classified it (`hubspot_changes_overview` of the current master)

In [ ]:
ov_path = CUR.parent / f'hubspot_changes_overview_{month}.csv'
if ov_path.exists():
    ov = pd.read_csv(ov_path, dtype=str, keep_default_na=False)
    print(ov['change'].value_counts().to_dict())
    print('RETIRED per ÜO:', ov[ov['change'] == 'retired']['uo'].value_counts().to_dict())
    display(ov[ov['fields'].str.startswith('duplicate of')])  # duplicates closed as CHANGED
else:
    print('no overview next to', CUR)